In [1]:
import sys
import os
import random
import torch

repo_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
sys.path.insert(0, repo_root)

from peptide_pipeline.generator.cvae_generator import CVAEGenerator


In [ ]:
repo_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
sys.path.insert(0, repo_root)


# Import the JSONDataLoader class from the dataloader_json.py in dataloader module
from peptide_pipeline.dataloader.dataloader_json import DataLoader as JSONDataLoader
# Example usage
json_loader = JSONDataLoader()
json_loader.load_data("ai_training_peptides.json", columns=["sequence","length","cathionicity","hydrophobicity"])
print(f"Loaded {len(json_loader.data)} peptides from JSON file")
data = json_loader.get_data()

filtered_data = data[(data["length"] >= 5) & (data["length"] <= 14)]
print(f"Filtered peptides (length 5-14): {len(filtered_data)}")

# get min max for each property
properties = ["length","cathionicity","hydrophobicity"]
min_max_values = {}
for prop in properties:
    min_val = filtered_data[prop].min()
    max_val = filtered_data[prop].max()
    min_max_values[prop] = {"min": min_val, "max": max_val}
    print(f"{prop}: min={min_val}, max={max_val}")

constraints_default = {
    "size": {"min": min_max_values["length"]["min"], "max": min_max_values["length"]["max"]},
    "molecular_weight": {"min": 500, "max": 3000},
    "net_charge_pH5_5": {"min": -5, "max": 5},
    "isoelectric_point": {"min": 3, "max": 11},
    "hydrophobicity": {"min": min_max_values["hydrophobicity"]["min"], "max": min_max_values["hydrophobicity"]["max"]},
    "cathionicity": {"min": min_max_values["cathionicity"]["min"], "max": min_max_values["cathionicity"]["max"]},
    "hydrophobic_moment": {"min": 0, "max": 1},
    "logp": {"min": -3, "max": 3},
    "boman_index": {"min": -5, "max": 5},
    "h_bond_donors": {"min": 0, "max": 10},
    "h_bond_acceptors": {"min": 0, "max": 10},
    "tpsa": {"min": 0, "max": 200},
}

#  --- Initialize model ---
cvae = CVAEGenerator( latent_dim=16, hidden_dim=64,condition_dim= 32)
print(f"Device:     {cvae.device}")
print(f"Input dim:  {cvae.input_dim}")
print(f"Latent dim: {cvae.latent_dim}")
print(f"Hidden dim: {cvae.hidden_dim}")
print(f"Condition dim: {cvae.condition_dim}")

conditions = cvae._constraints_to_condition_tensor(
    constraints=constraints_default,
    count=1,
    device=cvae.device,
)
print(f"Condition tensor shape: {conditions.shape}")

MAX_LEN = 14
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
aa_index = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
PAD_IDX = 20
VOCAB_SIZE = 21

# keep range instead of fixed len
sequences = [s for s in filtered_data["sequence"].tolist() if 5 <= len(s) <= MAX_LEN]
lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long, device=cvae.device)

def encode_with_pad(seqs, max_len):
    x = torch.zeros(len(seqs), max_len * VOCAB_SIZE)
    for i, seq in enumerate(seqs):
        # PAD all positions first
        for j in range(max_len):
            x[i, j * VOCAB_SIZE + PAD_IDX] = 1.0
        # overwrite valid positions with AA one-hot
        for j, aa in enumerate(seq[:max_len]):
            if aa in aa_index:
                x[i, j * VOCAB_SIZE + PAD_IDX] = 0.0
                x[i, j * VOCAB_SIZE + aa_index[aa]] = 1.0
    return x

x = encode_with_pad(sequences, MAX_LEN).to(cvae.device)

conditions = cvae._constraints_to_condition_tensor(
    constraints=constraints_default,
    count=len(sequences),  # must match batch dimension of x
    device=cvae.device
)

cvae = CVAEGenerator(max_len=MAX_LEN, latent_dim=16, hidden_dim=64, condition_dim=32)
cvae.train_model(data=x, conditions=conditions, lengths=lengths, epochs=100, batch_size=64, lr=1e-3, kl_anneal_epochs=100)

Loaded 3653 peptides from JSON file
Filtered peptides (length 5-14): 3210
length: min=5, max=12
cathionicity: min=0, max=12
hydrophobicity: min=-3.89, max=4.5
Device:     cuda
Input dim:  294
Latent dim: 16
Hidden dim: 64
Condition dim: 32
Condition tensor shape: torch.Size([1, 32])


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


  Epoch 050/100 | Recon: 2.4582 | KL: 0.0000 | KL weight: 0.49
  Epoch 100/100 | Recon: 2.4546 | KL: 0.0000 | KL weight: 0.99


In [3]:
# --- Generate ---
generated = cvae.generate_peptides(count=10,constraints = constraints_default,temperature=0.8)
amino_acid_set = set(AMINO_ACIDS)
unique_generated = set()
for i, pep in enumerate(generated):
    valid = all(c in amino_acid_set for c in pep) and len(pep) == lengths.max().item()
    status = "✓" if valid else "✗ INVALID"
    unique_generated.add(pep)
    print(f"Peptide {i+1:02d}: {pep}  {status}")   

Peptide 01: VRRKKKLR  ✗ INVALID
Peptide 02: RLKRKWKIKLK  ✗ INVALID
Peptide 03: SLKWLRLVK  ✗ INVALID
Peptide 04: LIRHMNGIRLWS  ✓
Peptide 05: AKWRARLG  ✗ INVALID
Peptide 06: KRRRKVVA  ✗ INVALID
Peptide 07: LGLYLW  ✗ INVALID
Peptide 08: KLFKRRRLLK  ✗ INVALID
Peptide 09: PLLELILK  ✗ INVALID
Peptide 10: RIFRIRKSKI  ✗ INVALID


In [4]:
# test with clearly defined constraints and check if generated peptides meet those constraints
constraints = {
    "size": {"min": 12, "max": 12},
    "molecular_weight": {"min": 500, "max": 3000},
    "net_charge_pH5_5": {"min": 4, "max": 4},
    "isoelectric_point": {"min": 14, "max": 14},
    "hydrophobicity": {"min": 0.10, "max": 0.10},
    "cathionicity": {
        "min": min_max_values["cathionicity"]["min"],   # scalar, not [scalar]
        "max": min_max_values["cathionicity"]["max"],   # scalar, not [scalar]
    },
    "hydrophobic_moment": {"min": 0, "max": 1},
    "logp": {"min": -3, "max": 3},
    "boman_index": {"min": -5, "max": 5},
    "h_bond_donors": {"min": 0, "max": 10},
    "h_bond_acceptors": {"min": 0, "max": 10},
    "tpsa": {"min": 0, "max": 200},
}

generated = cvae.generate_peptides(
    count=10,
    constraints=constraints,   # pass dict here
    temperature=0.8
)

for i, pep in enumerate(generated, 1):
    print(f"{i:02d}. {pep}")

01. GRWLRWKWRLRR
02. KKKRQIKLWIGK
03. GLIFLILWVIKR
04. IKGLARRKKRRR
05. RKFRRLAKRWKW
06. CREALRKALKIK
07. VWWKWWILRSKK
08. RLWLRWGTHKRN
09. GKKVWRGTLLLI
10. GRIAYWPLWAKR
